# Lingustics of a projection

In [1]:
# --- spaCy (Morphologizer) ---

import spacy

future_tense = []

nlp = spacy.load("en_core_web_lg")
doc = nlp("He will go running.")
for t in doc:
    tense = t.morph.get("Tense")
    
    if tense == ['Futu']:
        future_tense.append(tense)

    print("spaCy:", t.text, t.morph.get("Tense"))


spaCy: He []
spaCy: will []
spaCy: go []
spaCy: running ['Pres']
spaCy: . []


In [2]:
import spacy
nlp = spacy.load("en_core_web_lg")
doc = nlp("He will go running.")

for t in doc:
    if t.tag_ == "MD" or (t.pos_ == "AUX" and t.lemma_ == "will"):
        print("Modal aux (future-ish):", t.text, {"pos": t.pos_, "tag": t.tag_, "morph": t.morph, "dep": t.dep_})

# Optional: flag the clause as 'future' if MD precedes a base verb
has_future = any(t.tag_ == "MD" and t.head.tag_ in {"VB", "VBP"} for t in doc)
print("future_like_clause:", has_future)

Modal aux (future-ish): will {'pos': 'AUX', 'tag': 'MD', 'morph': VerbForm=Fin, 'dep': 'aux'}
future_like_clause: True


In [3]:
import os
import sys
import warnings
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.datasets import make_classification
from imblearn.over_sampling import RandomOverSampler, SMOTE

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_visualizing import DataVisualizing
from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

## Load Dataset

In [4]:
df = DataProcessing.load_single_synthetic_data(
    notebook_dir, batch_idx=1, sep=',', return_as='dataframe'
)
df = df.loc[:7, :]
df

Loading: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/prediction_logs/batch_1-prediction/batch_1-from_df.csv


,Base Sentence,Sentence Label,Domain,Model Name,API Name,Batch ID,Template Number
0,JPMorgan Chase forecasts that the net profit a...,1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,1
1,"On August 21, 2024, Bank of America speculates...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,2
2,"Citigroup predicts on 2024-08-21, the operatin...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,3
3,"According to Goldman Sachs, the research and d...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,4
4,"In 21 August 2024, Morgan Stanley envisions th...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,5
5,The stock price at Visa should stay same in Q2...,1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,6
6,JPMorgan forecasts that the revenue at Microso...,1,finance,llama-3.3-70b-instruct,NAVI_GATOR,0,1
7,"On August 25, 2024, to September 25, 2025, Cit...",1,finance,llama-3.3-70b-instruct,NAVI_GATOR,0,2


In [5]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
chronicle2050_file = os.path.join(base_data_path, 'chronicle2050/data.csv')
df = DataProcessing.load_from_file(chronicle2050_file, sep=',')
df

,index,sentence,label
0,0,"By January 1st, 2037, Tesla will have been the...",1
1,1,An annual average temperature anomaly value ab...,1
2,2,Private Nonfarm business productivity growth w...,1
3,3,No Republican will be President of the USA bef...,1
4,4,The market capitalization of Berkshire Hathawa...,1
...,...,...,...
6397,2540,Many major technology players are [TeleNav Inc...,0
6398,2541,"WaterIQ Technologies, the leader in next-gener...",0
6399,2542,The Business Research Company's 'Clean Coal Te...,0
6400,2543,'Prophecy Market Insights offers a 20% discoun...,0


## Extract Tense

+ Morphology $ \rightarrow $ morphemes
    1. spaCy: 
        - Technique: Statistical tagger
        - Output: Feature values organized as key-value pairs (e.g., `Number=Sing|Tense=Past`)
        - Note: NOT a full derivational analysis; it reports surface features, not raw morphemes.
        - Note: It doesn’t rebuild the word from morphemes; it infers probabilistically the most likely label set.
        - Note: Doesn't check the construction of the word, instead utilizes probability to determine label/feature value
        - Note: `token.noun_chunks` is for phrase‑level grouping, not morpheme segmentation.
        - Note: `token._.morph_analysis` (via an extension) gives access to the same feature set, not a morpheme list.

    2. Morphological Analysis: 
        - Technique: Explicit morpheme splitter (prefixes, stems, suffixes, infixes, etc.)
        - Output: decomposed structure + #morphemes + phonetics of each morphome
        - Note: Full derivational analysis

In [6]:
disable_components = []
# spe = SpacyFeatureExtraction(df, 'Base Sentence')
spe = SpacyFeatureExtraction(df, 'sentence')
pos_df, ner_df = spe.extract_pos_ner_features(disable_components=[], visualize=True)
pos_df




EXTRACTING POS FEATURES


Extracting POS Features:   0%|          | 0/6402 [00:00<?, ?doc/s]


	####### Sentence (0): By January 1st, 2037, Tesla will have been the first company with 1 million vehicles that are capable of SAE Level 4 autonomy on over 90% of public roads in the contiguous United States, with human-level safety or better, and this capability will be usable by the general public commercially. #######


Extracting POS Features:   0%|          | 1/6402 [00:00<33:02,  3.23doc/s]


	####### Sentence (1): An annual average temperature anomaly value above the 1850-1899 baseline will be published in the Berkeley Earth Global Temperature series as 2.0C or higher on or before the 2037 value (published in 2038). #######


...


	####### Sentence (2): Private Nonfarm business productivity growth will average over 1.8 percent per year from the first quarter (Q1) of 2020 to the last quarter of 2029 (Q4). #######



	####### Sentence (3): No Republican will be President of the USA before December 30, 2039. #######


Extracting NER Features: 100%|██████████| 6402/6402 [00:22<00:00, 279.54doc/s]



DONE EXTRACTING NER FEATURES


,Sentence,Term,Lemma,POS Label,Detailed POS Label,Dependency,Shape,Is Alpha,Stop Word,Unique POS Label,...,Definite,Degree,SubCat,Animacy,Acc,Gen,Loc,Ins,Parat,Prep
0,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",By,by,ADP,IN,prep,Xx,True,True,ADP_1,...,,,,,,,,,,
1,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",January,January,PROPN,NNP,compound,Xxxxx,True,False,PROPN_1,...,,,,,,,,,,
2,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",1st,1st,NOUN,NN,pobj,dxx,False,False,NOUN_1,...,,,,,,,,,,
3,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",",",",",PUNCT,",",punct,",",False,False,PUNCT_1,...,,,,,,,,,,
4,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",2037,2037,NUM,CD,appos,dddd,False,False,NUM_1,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189829,"(The, Northern, Virginia, Technology, Council,...",Technology,Technology,PROPN,NNP,compound,Xxxxx,True,False,PROPN_19582,...,,,,,,,,,,
189830,"(The, Northern, Virginia, Technology, Council,...",CFO,CFO,PROPN,NNP,compound,XXX,True,False,PROPN_19583,...,,,,,,,,,,
189831,"(The, Northern, Virginia, Technology, Council,...",Awards,Awards,PROPN,NNPS,pobj,Xxxxx,True,False,PROPN_19584,...,,,,,,,,,,
189832,"(The, Northern, Virginia, Technology, Council,...",.,.,PUNCT,.,punct,.,False,False,PUNCT_19647,...,,,,,,,,,,


In [7]:
ner_df

,Sentence,Term,NER Label,Unique NER Label,Start Char,End Char
0,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",January 1st,DATE,DATE_1,3,14
1,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",2037,CARDINAL,CARDINAL_1,16,20
2,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",Tesla,ORG,ORG_1,22,27
3,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",first,ORDINAL,ORDINAL_1,47,52
4,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",1 million,CARDINAL,CARDINAL_2,66,75
...,...,...,...,...,...,...
24220,"(The, Northern, Virginia, Technology, Council,...",NVTC,ORG,ORG_2577,42,46
24221,"(The, Northern, Virginia, Technology, Council,...",today,DATE,DATE_4582,119,124
24222,"(The, Northern, Virginia, Technology, Council,...",27th,ORDINAL,ORDINAL_397,157,161
24223,"(The, Northern, Virginia, Technology, Council,...",Greater Washington Technology,ORG,ORG_2578,169,198


## Filter by what gives future tense
1. POS: 
    1. VERB
        - Unique POS: 
            | Tag | Form | Example | Grammatical Role | Tense (Future)
            |-----|------|---------|-------------------|------|
            | **VB** | Base form | `sleep`, `run` | Infinitive or base form (e.g., “to sleep”, “I sleep”) | ❌
            | **VBD** | Past tense | `slept`, `ran` | Simple past (e.g., “She slept”) | ❌
            | **VBG** | Present participle / gerund | `sleeping`, `running` | Progressive or gerund (e.g., “I am sleeping”, “Sleeping is fun”) | ❌
            | **VBN** | Past participle | `slept`, `run` | Perfect or passive (e.g., “She has slept”, “The house was run”) | ❌
            | **VBP** | Non‑3rd‑person singular present | `sleep`, `run` | “I sleep”, “They run” | ❌
            | **VBZ** | 3rd‑person singular present | `sleeps`, `runs` | “He sleeps” (the ‑s ending) | ❌
    2. AUX
        - Unique POS: 
            POS Tag | Detailed POS Tag | Form | Example | Grammatical Role | Tense (Future)
            |-----|-----|------|---------|-------------------|------|
            | **AUX** | **MD** | Modal | will | Modal auxiliary indicating future | ✅
            | **AUX** | **MD** | Modal | shall | Modal auxiliary expressing future (especially in formal English) | ✅
            | **AUX** | **MD** | Modal | should | Modal that can express future obligation or advice | ✅
            | **AUX** | **MD** | Modal | may | Modal that can express future possibility | ✅
            | **AUX** | **MD** | Modal | might | Modal that can express future conditionality | ✅
            | **AUX** | **MD** | Modal | could | Modal that can express future potential | ✅

In [8]:
sentence_pos_df = pos_df.loc[:, ['Sentence', 'POS Label']]
filt_aux = (sentence_pos_df['POS Label'] == 'AUX')
sentence_pos_df[filt_aux]

,Sentence,POS Label
7,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",AUX
8,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",AUX
9,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",AUX
18,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",AUX
49,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",AUX
...,...,...
189556,"(Ev, Dynamics, (, Holdings, ), Limited, (, the...",AUX
189621,"(Many, major, technology, players, are, [, Tel...",AUX
189671,"(Many, major, technology, players, are, [, Tel...",AUX
189730,"(The, Business, Research, Company, 's, ', Clea...",AUX


## Filter by what gives future tense
- Morphology: Person

In [9]:
sentence_person_df = pos_df.loc[:, ['Sentence', 'Person']]
filt_person = (sentence_person_df['Person'].astype(str) == "['3']")
sentence_person_df[filt_person]

,Sentence,Person
447,"(There, will, be, at, least, one, 2022, FIFA, ...",[3]
582,"(“, The​, ​US​, ​unemployment​, ​rate,​, ​as​,...",[3]
727,"(During, the, 500, years, between, 02015, and,...",[3]
728,"(During, the, 500, years, between, 02015, and,...",[3]
751,"(Past, performance, would, suggest, 3, -, 4, %...",[3]
...,...,...
189737,"(The, Business, Research, Company, 's, ', Clea...",[3]
189752,"(', Prophecy, Market, Insights, offers, a, 20,...",[3]
189768,"(', Prophecy, Market, Insights, offers, a, 20,...",[3]
189772,"(', Prophecy, Market, Insights, offers, a, 20,...",[3]


## Filter by what gives future tense

- Morphology: Tense

In [10]:
sentence_tense_df = pos_df.loc[:, ['Sentence', 'Tense']]
filt_person = (sentence_tense_df['Tense'].astype(str) == "['Pres']")
sentence_tense_df[filt_person]

,Sentence,Tense
18,"(By, January, 1st, ,, 2037, ,, Tesla, will, ha...",[Pres]
189,"(By, 2035, there, will, be, at, least, 10, peo...",[Pres]
215,"(Between, 2021, and, the, end, of, 2030, ,, an...",[Pres]
226,"(Between, 2021, and, the, end, of, 2030, ,, an...",[Pres]
281,"(Over, a, ten, -, year, period, commencing, on...",[Pres]
...,...,...
189737,"(The, Business, Research, Company, 's, ', Clea...",[Pres]
189752,"(', Prophecy, Market, Insights, offers, a, 20,...",[Pres]
189768,"(', Prophecy, Market, Insights, offers, a, 20,...",[Pres]
189772,"(', Prophecy, Market, Insights, offers, a, 20,...",[Pres]


In [11]:
df

,index,sentence,label
0,0,"By January 1st, 2037, Tesla will have been the...",1
1,1,An annual average temperature anomaly value ab...,1
2,2,Private Nonfarm business productivity growth w...,1
3,3,No Republican will be President of the USA bef...,1
4,4,The market capitalization of Berkshire Hathawa...,1
...,...,...,...
6397,2540,Many major technology players are [TeleNav Inc...,0
6398,2541,"WaterIQ Technologies, the leader in next-gener...",0
6399,2542,The Business Research Company's 'Clean Coal Te...,0
6400,2543,'Prophecy Market Insights offers a 20% discoun...,0


In [12]:
sentence_person_df['Tense'].value_counts()

KeyError: 'Tense'

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_lg")
doc = nlp("Rats are various medium-sized, long-tailed rodents.")
displacy.render(doc, style="dep", jupyter=True)


In [ ]:
from IPython.display import HTML, display
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_lg")
doc = nlp("Rats are various medium-sized, long-tailed rodents.")

# Get HTML string and display it
html = displacy.render(doc, style="dep", jupyter=False)
display(HTML(html))

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_lg")
doc1 = nlp("This is a sentence.")
doc2 = nlp("This is another sentence.")
html = displacy.render([doc1, doc2], style="dep", page=True)